# Practice: Earthquake Sources, Spectra, and Magnitudes

> **Colab note:** This notebook is designed to run on **Google Colab**. The first code cell installs dependencies. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS164/blob/main/notebooks/09_source_theory_practice.ipynb)

### Learning Objectives:

- Interpret basic focal mechanism and beachball patterns
- Calculate scalar seismic moment and moment magnitude
- Relate a source time function $M(t)$ to far-field displacement $\dot{M}(t)$
- Explore how rupture duration controls corner frequency and spectral shape
- Calculate directivity effects for a unilateral rupture
- Estimate stress drop from moment and rupture size
- Distinguish fracture energy from frictional dissipation using a stress-slip curve
- Explain magnitude saturation using the omega-square source spectrum

**Prerequisites:** Basic Python, logarithms, unit conversions, and introductory earthquake source concepts

**Reference:** Shearer, Chapter 9 (Earthquakes and Source Theory)

**Notebook Outline:**

- [Part (a): Focal Mechanisms and Beachballs](#Part-a-Focal-Mechanisms-and-Beachballs)
- [Part (b): Scalar Seismic Moment and Moment Magnitude](#Part-b-Scalar-Seismic-Moment-and-Moment-Magnitude)
- [Part (c): Source Spectra and Corner Frequency](#Part-c-Source-Spectra-and-Corner-Frequency)
- [Part (d): Rupture Directivity](#Part-d-Rupture-Directivity)
- [Part (e): Magnitude Saturation](#Part-e-Magnitude-Saturation)
- [Summary](#Summary)

In [36]:
# Install dependencies (for Google Colab or missing packages)
import sys

try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running in local environment")

required_packages = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
    'pandas': 'pandas',
    'obspy': 'obspy'
}

missing_packages = []
for package, pip_name in required_packages.items():
    try:
        __import__(package)
        print(f"OK: {package} is already installed")
    except ImportError:
        missing_packages.append(pip_name)
        print(f"MISSING: {package} not found")

if missing_packages:
    print(f"\nInstalling missing packages: {', '.join(missing_packages)}")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_packages)
    print("Installation complete")
else:
    print("\nAll required packages are installed")

Running in local environment
OK: numpy is already installed
OK: matplotlib is already installed
OK: ipywidgets is already installed
OK: pandas is already installed
OK: obspy is already installed

All required packages are installed


In [37]:
# Imports
import matplotlib.pyplot as plt
import numpy as np

---

# Part (a): Focal Mechanisms and Beachballs

Beachball diagrams summarize the P-wave first-motion radiation pattern of an earthquake.

**Tasks**:

In this exercise you will:

1. Compute P-wave takeoff angles
2. Plot first-motion polarity observations on the focal sphere
3. Compare observations to a focal mechanism
4. Interpret the resulting beachball

In [38]:
import numpy as np
import pandas as pd

source_depth_km = 8

stations = pd.DataFrame({

    "station": [
        "STA001","STA002","STA003","STA004","STA005",
        "STA006","STA007","STA008","STA009","STA010",
        "STA011","STA012","STA013","STA014","STA015",
        "STA016","STA017","STA018","STA019","STA020",
        "STA021","STA022","STA023","STA024","STA025",
        "STA026","STA027","STA028","STA029","STA030",
        "STA031","STA032","STA033","STA034","STA035",
        "STA036"
    ],

    # azimuth clockwise from north
    "azimuth": np.array([

          0,  10,  20,  30,  40,
         50,  60,  70,  80,  90,
        100, 110, 120, 130, 140,
        150, 160, 170, 180, 190,
        200, 210, 220, 230, 240,
        250, 260, 270, 280, 290,
        300, 310, 320, 330, 340,
        350,

    ]),

    # varied local/regional distances
    "distance_km": np.array([

         15,  20,  28,  35,  45,
         55,  65,  75,  85,  95,
         90,  80,  70,  60,  50,
         40,  30,  22,  18,  25,
         35,  48,  62,  78,  92,
         88,  74,  58,  42,  30,
         20,  16,  24,  38,  52,
         68,

    ]),

    "polarity": np.array([

        -1, -1, -1, -1, -1,
        1,  1,  1, 1,  1,
         1,  1,  1,  1,  1,
         -1, -1, -1, -1, -1,
        -1, -1, -1, -1,  -1,
         1,  1,  1, 1,  1,
         1,  1, 1, -1, -1,
         -1,

    ])
})

stations

,station,azimuth,distance_km,polarity
0,STA001,0,15,-1
1,STA002,10,20,-1
2,STA003,20,28,-1
3,STA004,30,35,-1
4,STA005,40,45,-1
5,STA006,50,55,-1
6,STA007,60,65,1
7,STA008,70,75,1
8,STA009,80,85,1
9,STA010,90,95,1


## Compute Takeoff Angles

For a simple 1D Earth model, the P-wave takeoff angle can be approximated using ray geometry:

$$
\theta_t = \tan^{-1}\left(\frac{\Delta}{h}\right)
$$

where:
- $\theta_t$ = takeoff angle
- $\Delta$ = horizontal source-receiver distance
- $h$ = earthquake depth

For this exercise:
- distance is already provided in km
- earthquake depth is 8 km

This approximation assumes straight-ray propagation in a homogeneous medium.

In [39]:
# compute takeoff angles

takeoff_angles =

stations["takeoff_angle"] = takeoff_angles

stations

,station,azimuth,distance_km,polarity,takeoff_angle
0,STA001,0,15,-1,61.927513
1,STA002,10,20,-1,68.198591
2,STA003,20,28,-1,74.054604
3,STA004,30,35,-1,77.124998
4,STA005,40,45,-1,79.919402
5,STA006,50,55,-1,81.724107
6,STA007,60,65,1,82.983498
7,STA008,70,75,1,83.911472
8,STA009,80,85,1,84.623295
9,STA010,90,95,1,85.186449


## Plot First Motions on the Focal Sphere

Filled circles = compressional first motions

Open circles = dilatational first motions

In [40]:
import numpy as np
import matplotlib.pyplot as plt

from obspy.imaging.beachball import beach
from ipywidgets import interact, IntSlider

# ---------------------------------------------------
# Force light plotting theme
# ---------------------------------------------------

plt.style.use("default")

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

# ---------------------------------------------------
# Equal-area lower hemisphere projection
# ---------------------------------------------------

def focal_projection(takeoff, azimuth):

    takeoff_rad = np.radians(takeoff)
    azimuth_rad = np.radians(azimuth)

    r = np.sqrt(2) * np.sin(takeoff_rad / 2)

    x = r * np.sin(azimuth_rad)
    y = r * np.cos(azimuth_rad)

    return x, y


# ---------------------------------------------------
# Interactive beachball fitting
# ---------------------------------------------------

def plot_focal_mechanism(strike=320, dip=90, rake=180):

    fig, ax = plt.subplots(figsize=(8,8))

    # draw beachball
    b = beach(
        [strike, dip, rake],
        xy=(0,0),
        width=2,
        linewidth=1.5,
        facecolor='k',
        edgecolor='k',
        alpha=0.85
    )

    ax.add_collection(b)

    # plot observations
    for _, row in stations.iterrows():

        x, y = focal_projection(
            row["takeoff_angle"],
            row["azimuth"]
        )

        observed = row["polarity"]

        if observed == 1:

            ax.plot(
                x, y,
                'ko',
                markersize=7,
                zorder=0
            )

        else:

            ax.plot(
                x, y,
                'wo',
                markersize=7,
                markeredgecolor='k',
                zorder=0
            )

    # cardinal directions
    ax.text(0, 1.08, "N", fontsize=14, ha='center')
    ax.text(1.08, 0, "E", fontsize=14, va='center')
    ax.text(0, -1.08, "S", fontsize=14, ha='center')
    ax.text(-1.08, 0, "W", fontsize=14, va='center')

    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)

    ax.set_aspect('equal')

    ax.set_title(
        f"Strike = {strike}°   Dip = {dip}°   Rake = {rake}°",
        fontsize=14
    )

    ax.axis('off')

    plt.show()


# ---------------------------------------------------
# Interactive sliders
# ---------------------------------------------------

interact(
    plot_focal_mechanism,

    strike=IntSlider(
        min=0,
        max=360,
        step=1,
        value=320,
        description='Strike'
    ),

    dip=IntSlider(
        min=0,
        max=90,
        step=1,
        value=90,
        description='Dip'
    ),

    rake=IntSlider(
        min=-180,
        max=180,
        step=1,
        value=180,
        description='Rake'
    )
)

interactive(children=(IntSlider(value=320, description='Strike', max=360), IntSlider(value=90, description='Di…

<function __main__.plot_focal_mechanism(strike=320, dip=90, rake=180)>

**Tasks:**

1. Use the sliders to determine the strike, dip, and rake that best fit the polarity observations. Record your preferred focal mechanism solution below:

| Parameter | Estimated Value |
|---|---|
| Strike | |
| Dip | |
| Rake | |

2. Identify the two possible nodal planes.

3. Explain why first-motion polarity observations alone cannot distinguish the true fault plane from the auxiliary plane.

4. List one or two additional observations that could help identify the true fault plane.

---

## Part (b): Scalar Seismic Moment and Moment Magnitude

The scalar seismic moment is:

$$
M_0 = \mu D A
$$

where:

- $\mu$ is shear modulus
- $D$ is average slip
- $A$ is rupture area

Assume an earthquake has:

- rupture length: 100 km
- rupture width: 12 km
- average slip: 6 m
- shear modulus: 30 GPa

**Tasks:**

1. Convert all quantities to SI units
2. Calculate rupture area
3. Calculate scalar seismic moment in N m
4. Convert moment to dyne-cm
5. Calculate moment magnitude using:

$$
M_w = \frac{2}{3}\log_{10}(M_0) - 10.7
$$

where $M_0$ is in dyne-cm.

In [41]:
# Given values
length_km =     # km
width_km =      # km
slip_m =        # m
mu_GPa =        # GPa

# Convert to SI units
length_m =      # m
width_m =       # m
mu_Pa =         # Pa

# Rupture area
area_m2 =       # m^2

# Scalar seismic moment in N m
M0_Nm =         # N m

# Convert N m to dyne-cm
# 1 N m = 10^7 dyne-cm
M0_dyne_cm =    # dyne-cm

# Moment magnitude using dyne-cm formulation
Mw =            # dimensionless

print('Scalar Moment and Magnitude:')
print(f'  Area = {area_m2:.3e} m^2')
print(f'  M0   = {M0_Nm:.3e} N m')
print(f'  M0   = {M0_dyne_cm:.3e} dyne-cm')
print(f'  Mw   = {Mw:.2f}')

SyntaxError: invalid syntax (2089747860.py, line 2)

**Follow-up:** If average slip increases from 6 m to 8 m, how do $M_0$ and $M_w$ change?

In [ ]:
# New slip value
slip_new_m =       # m

# Recalculate moment and magnitude
M0_new_Nm =        # N m
M0_new_dyne_cm =   # dyne-cm
Mw_new =           # dimensionless

print('Effect of Increasing Slip:')
print(f'  Original M0 = {M0_Nm:.3e} N m, Mw = {Mw:.2f}')
print(f'  New M0      = {M0_new_Nm:.3e} N m, Mw = {Mw_new:.2f}')
print(f'  Moment ratio = {M0_new_Nm/M0_Nm:.2f}')
print(f'  Magnitude increase = {Mw_new - Mw:.2f}')

---

## Part (c): Source Spectra and Corner Frequency

A simple omega-square source spectrum is:

$$
A(f) = \frac{M_0}{1 + (f/f_c)^2}
$$

where:

- $M_0$ controls the low-frequency plateau
- $f_c$ is the corner frequency
- amplitudes decay as $f^{-2}$ at high frequency

**Tasks:**

1. Define a frequency vector
2. Calculate spectra for several corner frequencies
3. Plot the spectra on log-log axes
4. Explain how changing $f_c$ changes the pulse duration

In [ ]:
# Frequency vector
f = np.logspace(-3, 2, 1000)  # Hz

# Corner frequencies to compare
fc_values = [     ]  # Hz, e.g., [0.1, 1.0, 10.0]

plt.figure(figsize=(8, 5))
for fc in fc_values:
    A_f =          # omega-square spectrum
    A_f_norm =     # normalize by maximum amplitude for plotting
    plt.loglog(f, A_f_norm, label=f'fc = {fc} Hz')

plt.xlabel('Frequency (Hz)')
plt.ylabel('Normalized amplitude')
plt.title('Omega-square source spectra')
plt.legend()
plt.show()

**Conceptual question:**

Which source has a longer duration: one with high $f_c$ or one with low $f_c$?

*Your answer here:*

---

## Part (d): Rupture Directivity

For a unilateral rupture of length $L$ propagating at rupture velocity $v_r$, the apparent source duration depends on station azimuth:

$$
\tau_{\rm app}(\theta) = L\left(\frac{1}{v_r} - \frac{\cos\theta}{c}\right)
$$

where:

- $L$ is rupture length
- $v_r$ is rupture velocity
- $c$ is seismic wave speed
- $\theta$ is the station angle relative to rupture direction

Forward directivity shortens the pulse and increases its amplitude. Backward directivity lengthens the pulse and decreases its amplitude.

**Tasks:**

1. Calculate apparent duration for stations in the forward, side, and backward directions
2. Plot apparent duration as a function of azimuth
3. Explain why the pulse amplitude changes if the total moment is fixed

In [ ]:
# Rupture and wave speeds
L_km =       # rupture length in km
vr_km_s =    # rupture velocity in km/s
c_km_s =     # seismic wave speed in km/s

# Station angles
theta_deg = np.linspace(0, 180, 181)
theta_rad = np.deg2rad(theta_deg)

# Apparent duration
tau_app =    # seconds

# Special cases
tau_forward =    # theta = 0 degrees
tau_side =       # theta = 90 degrees
tau_backward =   # theta = 180 degrees

print('Apparent source durations:')
print(f'  Forward direction:  {tau_forward:.2f} s')
print(f'  Side direction:     {tau_side:.2f} s')
print(f'  Backward direction: {tau_backward:.2f} s')

plt.figure(figsize=(8, 4))
plt.plot(theta_deg, tau_app)
plt.xlabel('Station angle from rupture direction (degrees)')
plt.ylabel('Apparent duration (s)')
plt.title('Rupture directivity')
plt.show()

**Interpretation:**

Why are forward-directivity pulses shorter but larger in amplitude?

*Your answer here:*

---

## Part (e): Magnitude Saturation

Different magnitude scales sample different frequency bands.

Approximate examples:

- $M_L$ and $m_b$: short periods near 1 s, or about 1 Hz
- $M_s$: surface waves near 20 s, or about 0.05 Hz
- $M_w$: long-period moment, related to the low-frequency spectral plateau

As earthquakes get larger, rupture duration increases and corner frequency decreases. Fixed-frequency measurements eventually sample the high-frequency falloff instead of the flat low-frequency plateau.

This causes magnitude saturation.

**Tasks:**

1. Generate omega-square spectra for earthquakes from $M_w$ 4 to 9
2. Mark the approximate $m_b/M_L$ and $M_s$ measurement frequencies
3. Explain why $M_w$ does not saturate but $m_b$ and $M_L$ do

In [1]:
def M0_from_Mw(Mw):
    """Return scalar moment in N m from moment magnitude."""
    return 10**(1.5 * (Mw + 6.07))

# Constants for a simple constant-stress-drop scaling
beta =        # shear-wave velocity in m/s
k =           # source model constant, e.g., 0.21
stress_drop = # Pa, e.g., 3e6

# Frequencies
f = np.logspace(-3, 2, 1000)
Mw_values = [4, 5, 6, 7, 8, 9]

plt.figure(figsize=(8, 5))
for Mw_i in Mw_values:
    M0_i = M0_from_Mw(Mw_i)
    r_i =              # rupture radius from circular stress-drop equation
    fc_i =             # corner frequency from r = k beta / fc
    A_i =              # omega-square spectrum
    A_i_norm = A_i / M0_from_Mw(4)  # arbitrary plotting normalization
    plt.loglog(f, A_i_norm, label=f'Mw {Mw_i}, fc={fc_i:.3f} Hz')

plt.axvline(1.0, linestyle='--', color='black', label='~1 Hz: ML, mb')
plt.axvline(0.05, linestyle=':', color='black', label='~0.05 Hz: Ms')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude proxy')
plt.title('Magnitude saturation and the omega-square spectrum')
plt.legend(fontsize=8)
plt.show()

SyntaxError: invalid syntax (2987632899.py, line 6)

**Interpretation:**

Why do short-period magnitudes saturate for large earthquakes?

*Your answer here:*

Why does moment magnitude avoid this problem?

*Your answer here:*

---

## Summary

In this notebook, you practiced connecting earthquake source physics to seismic observations:

- Beachballs summarize radiation patterns and faulting style
- Seismic moment links slip, rupture area, and rigidity
- Source time functions control pulse shape and frequency content
- Directivity changes apparent pulse duration and amplitude
- Magnitude saturation occurs because fixed-frequency measurements stop tracking total moment for large events

**Big idea:** Seismograms record both the geometry and time dependence of earthquake rupture.